
# KKBox C-Level BI Dashboard Notebook

Notebook này được thiết kế để **đọc trực tiếp các output của `features-prep.ipynb`** và dựng bộ phân tích mô tả/phục vụ dashboard cho C-Level theo 2 lớp:

1. **Snapshot theo tháng mục tiêu** (`target_month`, `target_year`)
2. **Trend theo giai đoạn** (`month_start`, `year_start` → `month_end`, `year_end`)

Toàn bộ logic bên dưới **không quay lại raw CSV** mà chỉ dùng feature store đã được pipeline `features-prep.ipynb` xuất ra.


## Import necessary libraries

In [1]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, Markdown, display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
px.defaults.template = 'plotly_white'


## Configuration

> Chỉ cần chỉnh các biến ở cell dưới đây để đổi **tháng snapshot** và **giai đoạn trend**.


In [2]:

# # =========================
# # Paths
# # =========================
# PROJECT_DIR = Path.cwd()
# FEATURE_STORE_DIR = PROJECT_DIR / 'artifacts' / 'feature_store'
# BI_MASTER_PATH = FEATURE_STORE_DIR / 'bi_feature_master.parquet'
# TRAIN_BI_PATH = FEATURE_STORE_DIR / 'train_features_bi_all.parquet'
# TEST_BI_GLOB = 'test_features_bi_*_full.parquet'
# FEATURE_COLUMNS_PATH = FEATURE_STORE_DIR / 'feature_columns.csv'
# BI_DIMENSIONS_PATH = FEATURE_STORE_DIR / 'bi_dimension_columns.csv'

# # =========================
# # Snapshot month
# # =========================
# target_year = 2017
# target_month = 3

# # =========================
# # Trend range
# # =========================
# year_start = 2017
# month_start = 1
# year_end = 2017
# month_end = 4

# # =========================
# # Business rules / thresholds
# # =========================
# NEW_TENURE_DAYS = 30
# LOYAL_TENURE_DAYS = 365
# RECENCY_SAFE_DAYS = 21
# HIGH_SKIP_THRESHOLD = 0.50
# LOW_DISCOVERY_THRESHOLD = 0.20
# HIGH_VALUE_AMOUNT = 149.0
# HISTORICAL_CANCEL_RISK = 0.30
# SAFE_BASE_MAX_RISK_SCORE = 1

# # Optional: compact or wider charts
# DEFAULT_HEIGHT = 520
# WIDE_HEIGHT = 620


In [3]:
from pathlib import Path
import pandas as pd

# =========================
# Paths
# =========================
PROJECT_DIR = Path.cwd()

# Kaggle Input path
FEATURE_STORE_DIR = Path("/kaggle/input/datasets/ntthduong/kkbox-mock/feature_store")

if not FEATURE_STORE_DIR.exists():
    raise FileNotFoundError(f"Không tìm thấy thư mục: {FEATURE_STORE_DIR}")

BI_MASTER_PATH = FEATURE_STORE_DIR / "bi_feature_master.parquet"
TRAIN_BI_PATH = FEATURE_STORE_DIR / "train_features_bi_all.parquet"
TEST_BI_PATH = FEATURE_STORE_DIR / "test_features_bi_201704_full.parquet"
FEATURE_COLUMNS_PATH = FEATURE_STORE_DIR / "feature_columns.csv"
BI_DIMENSIONS_PATH = FEATURE_STORE_DIR / "bi_dimension_columns.csv"

# kiểm tra file tồn tại
required_files = [
    BI_MASTER_PATH,
    TRAIN_BI_PATH,
    TEST_BI_PATH,
    FEATURE_COLUMNS_PATH,
    BI_DIMENSIONS_PATH,
]

missing_files = [str(p) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "Thiếu các file sau:\n" + "\n".join(missing_files)
    )

print("PROJECT_DIR      :", PROJECT_DIR)
print("FEATURE_STORE_DIR:", FEATURE_STORE_DIR)
print("BI_MASTER_PATH   :", BI_MASTER_PATH)
print("TRAIN_BI_PATH    :", TRAIN_BI_PATH)
print("TEST_BI_PATH     :", TEST_BI_PATH)
print("FEATURE_COLUMNS  :", FEATURE_COLUMNS_PATH)
print("BI_DIMENSIONS    :", BI_DIMENSIONS_PATH)

# =========================
# Snapshot month
# =========================
target_year = 2017
target_month = 3

# =========================
# Trend range
# =========================
year_start = 2017
month_start = 1
year_end = 2017
month_end = 4

# =========================
# Business rules / thresholds
# =========================
NEW_TENURE_DAYS = 30
LOYAL_TENURE_DAYS = 365
RECENCY_SAFE_DAYS = 21
HIGH_SKIP_THRESHOLD = 0.50
LOW_DISCOVERY_THRESHOLD = 0.20
HIGH_VALUE_AMOUNT = 149.0
HISTORICAL_CANCEL_RISK = 0.30
SAFE_BASE_MAX_RISK_SCORE = 1

# Optional: compact or wider charts
DEFAULT_HEIGHT = 520
WIDE_HEIGHT = 620

PROJECT_DIR      : /kaggle/working
FEATURE_STORE_DIR: /kaggle/input/datasets/ntthduong/kkbox-mock/feature_store
BI_MASTER_PATH   : /kaggle/input/datasets/ntthduong/kkbox-mock/feature_store/bi_feature_master.parquet
TRAIN_BI_PATH    : /kaggle/input/datasets/ntthduong/kkbox-mock/feature_store/train_features_bi_all.parquet
TEST_BI_PATH     : /kaggle/input/datasets/ntthduong/kkbox-mock/feature_store/test_features_bi_201704_full.parquet
FEATURE_COLUMNS  : /kaggle/input/datasets/ntthduong/kkbox-mock/feature_store/feature_columns.csv
BI_DIMENSIONS    : /kaggle/input/datasets/ntthduong/kkbox-mock/feature_store/bi_dimension_columns.csv


## Help functions

In [4]:
def make_yyyymm(year: int, month: int) -> int:
    if month < 1 or month > 12:
        raise ValueError('month phải nằm trong [1, 12].')
    return int(year) * 100 + int(month)


def month_start_ts(yyyymm: int) -> pd.Timestamp:
    return pd.to_datetime(f'{yyyymm}01', format='%Y%m%d')


def month_label(yyyymm: int) -> str:
    return month_start_ts(yyyymm).strftime('%Y-%m')


def format_currency(value: float) -> str:
    return f'${value:,.0f}'


def format_pct(value: float) -> str:
    return f'{value:.1%}'


def safe_divide(numerator, denominator, default=0.0):
    if isinstance(numerator, (pd.Series, pd.Index)) or isinstance(denominator, (pd.Series, pd.Index)):
        numerator = pd.Series(numerator)
        denominator = pd.Series(denominator)
        out = numerator.div(denominator.replace(0, np.nan))
        return out.replace([np.inf, -np.inf], np.nan).fillna(default)
    if denominator in [0, None] or pd.isna(denominator):
        return default
    return numerator / denominator


def validate_period_inputs(year_start: int, month_start: int, year_end: int, month_end: int):
    start_period = make_yyyymm(year_start, month_start)
    end_period = make_yyyymm(year_end, month_end)
    if start_period > end_period:
        raise ValueError('Giai đoạn trend không hợp lệ: start period phải <= end period.')
    return start_period, end_period


def load_bi_feature_store() -> pd.DataFrame:
    if BI_MASTER_PATH.exists():
        df = pd.read_parquet(BI_MASTER_PATH)
        source_used = BI_MASTER_PATH
    else:
        frames = []
        if TRAIN_BI_PATH.exists():
            frames.append(pd.read_parquet(TRAIN_BI_PATH))
        test_candidates = sorted(FEATURE_STORE_DIR.glob(TEST_BI_GLOB))
        if test_candidates:
            test_df = pd.read_parquet(test_candidates[-1])
            if 'is_churn' not in test_df.columns:
                test_df['is_churn'] = np.nan
            frames.append(test_df)
        if not frames:
            raise FileNotFoundError(
                'Không tìm thấy bi_feature_master.parquet hoặc các file BI parquet fallback trong artifacts/feature_store.'
            )
        df = pd.concat(frames, ignore_index=True)
        source_used = 'fallback: train/test BI parquet'

    print(f'Loaded feature store from: {source_used}')
    return df


def quality_check_report(df: pd.DataFrame) -> pd.DataFrame:
    required_cols = [
        'msno', 'target_month', 'is_churn', 'expected_renewal_amount',
        'historical_transaction_rows', 'last_1_is_churn', 'membership_age_days',
        'days_since_last_listen', 'skip_ratio', 'discovery_ratio', 'is_auto_renew'
    ]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f'Thiếu các cột bắt buộc từ feature store: {missing_cols}')

    qc = pd.DataFrame({
        'metric': [
            'n_rows',
            'n_unique_users',
            'n_unique_target_months',
            'min_target_month',
            'max_target_month',
            'duplicate_msno_target_month',
            'null_is_churn',
            'null_expected_renewal_amount',
        ],
        'value': [
            len(df),
            df['msno'].nunique(),
            df['target_month'].nunique(),
            int(df['target_month'].min()),
            int(df['target_month'].max()),
            int(df.duplicated(subset=['msno', 'target_month']).sum()),
            int(df['is_churn'].isna().sum()),
            int(df['expected_renewal_amount'].isna().sum()),
        ]
    })
    return qc


def display_kpi_cards(card_items, n_cols=4):
    cards_html = []
    for title, value, subtitle in card_items:
        cards_html.append(
            f"""
            <div style='border:1px solid #E5E7EB; border-radius:16px; padding:16px; background:#FFFFFF; box-shadow:0 1px 3px rgba(0,0,0,0.05);'>
                <div style='font-size:13px; color:#6B7280; margin-bottom:6px;'>{title}</div>
                <div style='font-size:28px; font-weight:700; color:#111827; line-height:1.2;'>{value}</div>
                <div style='font-size:12px; color:#4B5563; margin-top:8px;'>{subtitle}</div>
            </div>
            """
        )
    grid = f"display:grid; grid-template-columns: repeat({n_cols}, minmax(0, 1fr)); gap:12px;"
    display(HTML(f"<div style='{grid}'>{''.join(cards_html)}</div>"))


def get_snapshot_slice(df: pd.DataFrame, target_year: int, target_month: int) -> pd.DataFrame:
    target_period = make_yyyymm(target_year, target_month)
    snapshot = df.loc[df['target_month'] == target_period].copy()
    if snapshot.empty:
        available = ', '.join(month_label(x) for x in sorted(df['target_month'].unique()))
        raise ValueError(f'Không có dữ liệu cho {target_period}. Các tháng hiện có: {available}')
    return snapshot


def get_trend_slice(df: pd.DataFrame, year_start: int, month_start: int, year_end: int, month_end: int) -> pd.DataFrame:
    start_period, end_period = validate_period_inputs(year_start, month_start, year_end, month_end)
    trend = df.loc[df['target_month'].between(start_period, end_period)].copy()
    if trend.empty:
        raise ValueError('Không có dữ liệu trong khoảng trend đã chọn.')
    return trend


def build_monthly_summary(df: pd.DataFrame) -> pd.DataFrame:
    monthly = (
        df.groupby(['target_month', 'month_label'], as_index=False)
          .agg(
              expiring_subscribers=('msno', 'nunique'),
              observable_active_users=('observable_active_flag', 'sum'),
              silent_expiring_users=('silent_expiring_flag', 'sum'),
              renewed_users=('renewed_flag', 'sum'),
              churned_users=('churned_flag', 'sum'),
              new_users=('new_user_flag', 'sum'),
              retained_users=('retained_user_flag', 'sum'),
              returning_users=('returning_user_flag', 'sum'),
              revenue_up_for_renewal=('revenue_up_for_renewal', 'sum'),
              safe_base_revenue=('safe_base_revenue', 'sum'),
              silent_expiring_revenue=('silent_expiring_revenue', 'sum'),
              revenue_at_risk_30d=('revenue_at_risk_30d', 'sum'),
              realized_lost_revenue_30d=('realized_lost_revenue_30d', 'sum'),
          )
          .sort_values('target_month')
          .reset_index(drop=True)
    )
    monthly['renewal_rate'] = safe_divide(monthly['renewed_users'], monthly['expiring_subscribers'])
    monthly['forward_churn_rate'] = safe_divide(monthly['churned_users'], monthly['expiring_subscribers'])
    monthly['expiring_cohort_growth_rate'] = (
        monthly['expiring_subscribers']
        .pct_change()
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    monthly['observable_active_share'] = safe_divide(
        monthly['observable_active_users'], monthly['expiring_subscribers']
    )
    monthly['revenue_growth_rate'] = (
        monthly['revenue_up_for_renewal']
        .pct_change()
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    monthly['risk_revenue_share'] = safe_divide(
        monthly['revenue_at_risk_30d'], monthly['revenue_up_for_renewal']
    )
    monthly['safe_base_share'] = safe_divide(
        monthly['safe_base_revenue'], monthly['revenue_up_for_renewal']
    )
    monthly['silent_revenue_share'] = safe_divide(
        monthly['silent_expiring_revenue'], monthly['revenue_up_for_renewal']
    )
    monthly['realized_lost_revenue_share'] = safe_divide(
        monthly['realized_lost_revenue_30d'], monthly['revenue_up_for_renewal']
    )
    monthly['arpu'] = safe_divide(monthly['revenue_up_for_renewal'], monthly['expiring_subscribers'])
    monthly['analysis_month'] = pd.to_datetime(monthly['target_month'].astype(str) + '01', format='%Y%m%d')
    return monthly


## Load data & quality check từ các output của `features-prep.ipynb`

Cell dưới đây ưu tiên đọc:

1. `artifacts/feature_store/bi_feature_master.parquet`
2. fallback sang `train_features_bi_all.parquet` + `test_features_bi_*.parquet`


In [5]:

raw_bi_df = load_bi_feature_store().copy()

# Chuẩn hoá kiểu dữ liệu cơ bản
raw_bi_df['target_month'] = raw_bi_df['target_month'].astype('int32')
raw_bi_df['msno'] = raw_bi_df['msno'].astype(str)

qc_report = quality_check_report(raw_bi_df)
display(qc_report)

available_months = sorted(raw_bi_df['target_month'].unique())
print('Available target months:', ', '.join(month_label(m) for m in available_months))

null_overview = (
    raw_bi_df[[
        'is_churn', 'expected_renewal_amount', 'membership_age_days',
        'days_since_last_listen', 'skip_ratio', 'discovery_ratio'
    ]]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename('null_pct')
    .reset_index()
    .rename(columns={'index': 'column'})
)

display(null_overview)


Loaded feature store from: /kaggle/input/datasets/ntthduong/kkbox-mock/feature_store/bi_feature_master.parquet


,metric,value
0,n_rows,44162
1,n_unique_users,14000
2,n_unique_target_months,4
3,min_target_month,201701
4,max_target_month,201704
5,duplicate_msno_target_month,0
6,null_is_churn,0
7,null_expected_renewal_amount,0


Available target months: 2017-01, 2017-02, 2017-03, 2017-04


,column,null_pct
0,is_churn,0.00
1,expected_renewal_amount,0.00
2,membership_age_days,0.00
3,days_since_last_listen,0.00
4,skip_ratio,0.00
5,discovery_ratio,0.00


## Feature engineering

Các biến dưới đây được tạo **trên lớp output của `features-prep.ipynb`** để chart phía dưới dùng trực tiếp, tránh lặp lại logic business ở mỗi cell.

### Nguyên tắc revenue cho dashboard C-Level

Notebook này đọc **expiring cohort** từ `features-prep.ipynb`, nên revenue được hiểu theo ba lớp:

1. **`revenue_up_for_renewal`**: toàn bộ doanh thu renewal amount đang đến kỳ gia hạnq trong tháng đó.  
2. **`safe_base_revenue`**: phần revenue đến từ nhóm expiring nhưng có tín hiệu giữ chân tốt hơn.  
3. **`revenue_at_risk_30d`**: phần revenue **có nguy cơ mất** trong 30 ngày tới theo business rule, **không dùng trực tiếp nhãn churn để dự báo**.  

Ngoài ra notebook có thêm **`realized_lost_revenue_30d`** để nhìn lại hậu kiểm lịch sử: doanh thu nào đã thật sự mất sau 30 ngày.

### Nhóm biến nhận diện cohort / snapshot

| Biến | Logic | Ý nghĩa business |
|---|---|---|
| `analysis_month` | chuyển `target_month` sang timestamp **cuối tháng** | dùng làm mốc snapshot của tháng target khi vẽ chart |
| `month_label` | format `YYYY-MM` | hiển thị nhãn cho chart và KPI |
| `revenue_up_for_renewal` | `clip(expected_renewal_amount, lower=0)` | doanh thu của **toàn bộ expiring cohort** đang bước vào kỳ gia hạn |
| `renewed_flag` | `is_churn == 0` | user đã gia hạn thành công trong 30 ngày sau expire |
| `churned_flag` | `is_churn == 1` | user không gia hạn trong cửa sổ 30 ngày |
| `observable_active_flag` | có `active_segment` khác `Inactive/Unknown`, hoặc fallback theo `count/listen_events_sum > 0` | user vẫn còn activity observable bên trong expiring cohort |
| `silent_expiring_flag` | phần bù của `observable_active_flag` | user expiring nhưng không còn dấu hiệu usage rõ ràng |

### Nhóm biến lifecycle / customer composition

| Biến | Logic | Ý nghĩa business |
|---|---|---|
| `new_user_flag` | `historical_transaction_rows <= 1` **hoặc** `membership_age_days < 30` | khách mới vào lifecycle trả phí |
| `returning_user_flag` | không phải new, `last_1_is_churn == 1`, hiện tại renewed | khách từng rời bỏ rồi quay lại |
| `retained_user_flag` | không phải new, không phải returning, hiện tại renewed | khách đang được giữ đều qua các cycle |
| `customer_composition` | ưu tiên gán theo thứ tự `Churned -> New -> Returning -> Retained` | phân rã cohort để đọc cơ cấu khách hàng tháng |
| `previous_customer_state` | dựa trên `historical_transaction_rows` và `last_1_is_churn` | trạng thái ở cycle trước, dùng cho Sankey movement |

### Nhóm biến risk / revenue logic

| Biến | Logic | Ý nghĩa business |
|---|---|---|
| `risk_score` | cộng điểm từ manual renew, recency xấu, skip cao, discovery thấp, cancel rate cao, churn history | thước đo short-term churn risk theo business rule |
| `risk_band` | map `risk_score` thành `Low / Watchlist / Medium / High` | gom mức độ cần theo dõi |
| `projected_risk_flag` | `risk_score >= 3` **hoặc** `silent_expiring_flag == 1` | user có nguy cơ rời bỏ làm mất revenue trong kỳ gia hạn tới |
| `safe_base_flag` | auto-renew, không silent, recency tốt, risk thấp | phần expiring cohort có khả năng giữ doanh thu tốt hơn |
| `safe_base_revenue` | `revenue_up_for_renewal * safe_base_flag` | phần doanh thu “an toàn” trong cohort đáo hạn |
| `silent_expiring_revenue` | `revenue_up_for_renewal * silent_expiring_flag` | doanh thu đang nằm ở nhóm ít/no usage observable |
| `revenue_at_risk_30d` | `revenue_up_for_renewal * projected_risk_flag` | **doanh thu rủi ro dự báo** trong 30 ngày tới |
| `realized_lost_revenue_30d` | `revenue_up_for_renewal * churned_flag` | **doanh thu đã thật sự mất** sau 30 ngày, dùng cho historical review |

### Nhóm biến segment cho dashboard

| Biến | Logic | Ý nghĩa business |
|---|---|---|
| `user_tier` | dựa trên `expected_renewal_amount` | phân tầng doanh thu để đọc risk theo tier |
| `high_value_flag` | `rfm_total_score` cao hoặc renewal amount lớn | khách hàng có giá trị tài chính cao |
| `loyal_flag` | tenure dài + lịch sử ổn định | nhóm trung thành, cần bảo vệ |
| `at_risk_flag` | dùng `projected_risk_flag` | nhóm cần can thiệp giữ chân trước khi outcome xảy ra |
| `customer_segment` | gán theo thứ tự `New -> At Risk -> Loyal -> High Value -> Core` | một nhãn segment thống nhất cho dashboard |


In [6]:
def add_dashboard_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Biến đổi feature store gốc thành bảng phân tích phục vụ BI dashboard cho C-Level.

    INPUT
    -----
    df : pd.DataFrame
        Bảng expiring-cohort output từ pipeline features-prep.
        Mỗi dòng thường đại diện cho 1 user nằm trong cohort sắp expire ở target_month.

    OUTPUT
    ------
    pd.DataFrame
        DataFrame đã được bổ sung:
        - cột thời gian phục vụ snapshot / trend
        - cột revenue theo logic "up for renewal"
        - cờ churn / renew
        - cờ observable active / silent
        - customer lifecycle / customer composition
        - rule-based risk scoring
        - safe base / risk revenue / realized lost revenue
        - user tier / customer segment
    """
    # =========================================================
    # 0) Copy dữ liệu để không sửa trực tiếp bảng đầu vào
    # =========================================================
    out = df.copy()

    # =========================================================
    # 1) Chuẩn hóa missing value
    # Trong pipeline trước đó, một số cột numeric dùng sentinel = -1
    # để biểu thị "không có dữ liệu" / "không quan sát được".
    # Ở lớp business rule này, -1 dễ bị hiểu sai là giá trị thật.
    # Vì vậy đổi -1 -> NaN cho các cột nhạy cảm trước khi áp rule.
    # =========================================================
    sentinel_cols = [
        "membership_age_days",
        "days_since_last_listen",
        "historical_cancel_rate",
        "recent_churn_events",
        "rfm_total_score",
        "count",
        "listen_events_sum",
        "expected_renewal_amount",
        "skip_ratio",
        "discovery_ratio",
        "last_1_is_churn",
        "historical_transaction_rows",
    ]

    for col in sentinel_cols:
        if col in out.columns:
            out[col] = out[col].replace(-1, np.nan)

    # =========================================================
    # 2) Tạo cột thời gian cho dashboard
    # target_month thường có dạng YYYYMM.
    # Dùng ngày cuối tháng để biểu diễn tháng target trên dashboard.
    # =========================================================
    out["analysis_month"] = (
        pd.to_datetime(out["target_month"].astype(str), format="%Y%m")
        + pd.offsets.MonthEnd(0)
    )
    out["month_label"] = out["analysis_month"].dt.strftime("%Y-%m")

    # =========================================================
    # 3) Revenue logic
    # QUAN TRỌNG: Đây là revenue của EXPIRING COHORT, KHÔNG phải full active base revenue.
    # =========================================================
    out["revenue_up_for_renewal"] = (
        out["expected_renewal_amount"].fillna(0).clip(lower=0)
    )
    
    # =========================================================
    # 4) Outcome flags
    # is_churn = 0 -> renew thành công / không churn
    # is_churn = 1 -> churn
    # =========================================================
    out["renewed_flag"] = (out["is_churn"] == 0).astype("int8")
    out["churned_flag"] = (out["is_churn"] == 1).astype("int8")

    # =========================================================
    # 5) Activity / observability logic
    # Mục tiêu:
    # - xác định user nào trong expiring cohort vẫn còn usage observable
    # - xác định user nào đã "silent"
    # =========================================================
    if "active_segment" in out.columns:
        activity_label = out["active_segment"].fillna("Unknown")

        # Observable active = còn có dấu hiệu usage
        observable_mask = activity_label.isin(
            ["Light 1-5 logs", "Active 6-15 logs", "Heavy > 15 logs"]
        )

        # Silent = inactive / unknown / không thuộc nhóm observable
        silent_mask = activity_label.isin(["Inactive", "Unknown"]) | (~observable_mask)
    else:
        # Fallback nếu không có active_segment
        observable_mask = (
            (out.get("count", pd.Series(0, index=out.index)).fillna(0) > 0)
            | (out.get("listen_events_sum", pd.Series(0, index=out.index)).fillna(0) > 0)
        )
        silent_mask = ~observable_mask

    out["observable_active_flag"] = observable_mask.astype("int8")
    out["silent_expiring_flag"] = silent_mask.astype("int8")

    # =========================================================
    # 6) Lifecycle / composition
    # =========================================================

    # New user:
    # - gần như chưa có nhiều transaction history
    # - hoặc tuổi thuê bao còn ngắn
    out["new_user_flag"] = (
        (out["historical_transaction_rows"].fillna(0) <= 1)
        | (out["membership_age_days"].fillna(np.inf) < NEW_TENURE_DAYS)
    ).astype("int8")

    # Returning:
    # - không phải new
    # - kỳ trước đã churn
    # - kỳ hiện tại renew được
    out["returning_user_flag"] = (
        (out["new_user_flag"] == 0)
        & (out["last_1_is_churn"].fillna(-1) == 1)
        & (out["renewed_flag"] == 1)
    ).astype("int8")

    # Retained:
    # - không phải new
    # - kỳ trước không churn
    # - kỳ hiện tại vẫn renew
    out["retained_user_flag"] = (
        (out["new_user_flag"] == 0)
        & (out["last_1_is_churn"].fillna(-1) == 0)
        & (out["renewed_flag"] == 1)
    ).astype("int8")

    # Gộp thành customer composition
    out["customer_composition"] = np.select(
        [
            out["churned_flag"] == 1,
            out["new_user_flag"] == 1,
            out["returning_user_flag"] == 1,
            out["retained_user_flag"] == 1,
        ],
        ["Churned", "New", "Returning", "Retained"],
        default="Unclassified",
    )

    # Trạng thái của khách hàng ở chu kỳ trước
    out["previous_customer_state"] = np.select(
        [
            out["historical_transaction_rows"].fillna(0) <= 1,
            out["last_1_is_churn"].fillna(-1) == 1,
            out["last_1_is_churn"].fillna(-1) == 0,
        ],
        ["First Cycle", "Previously Churned", "Previously Renewed"],
        default="Unknown",
    )

    # =========================================================
    # 7) Risk score
    # Đây là score theo BUSINESS RULE, không phải ML model.
    # Chỉ dùng tín hiệu TRƯỚC outcome, không dùng churned_flag để tránh leakage.
    # =========================================================
    risk_score = (
        # Không bật auto-renew -> rủi ro cao hơn
        (out["is_auto_renew"].fillna(0) == 0).astype(int)
        # Lâu rồi không nghe -> engagement thấp
        + (out["days_since_last_listen"].fillna(np.inf) > RECENCY_SAFE_DAYS).astype(int)
        # Skip nhiều -> trải nghiệm kém phù hợp
        + (out["skip_ratio"].fillna(0) >= HIGH_SKIP_THRESHOLD).astype(int)
        # Discovery thấp -> engagement yếu
        + (out["discovery_ratio"].fillna(1) < LOW_DISCOVERY_THRESHOLD).astype(int)
        # Từng cancel nhiều trong lịch sử
        + (
            out.get("historical_cancel_rate", pd.Series(0, index=out.index)).fillna(0)
            >= HISTORICAL_CANCEL_RISK
        ).astype(int)
        # Gần đây từng có churn event
        + (
            out.get("recent_churn_events", pd.Series(0, index=out.index)).fillna(0) >= 1
        ).astype(int)
    )

    out["risk_score"] = risk_score.astype("int16")

    # Chuyển score thành band dễ đọc hơn
    out["risk_band"] = np.select(
        [
            out["risk_score"] >= 4,
            out["risk_score"] >= 2,
            out["risk_score"] >= 1,
        ],
        ["High Risk", "Medium Risk", "Watchlist"],
        default="Low Risk",
    )

    # =========================================================
    # 8) Projected risk flag
    # User được xem là risk nếu:
    # - risk_score >= 3
    # hoặc
    # - user đang silent
    # =========================================================
    out["projected_risk_flag"] = (
        (out["risk_score"] >= 3) | (out["silent_expiring_flag"] == 1)
    ).astype("int8")

    # =========================================================
    # 9) Safe base flag
    # Phần expiring revenue có tín hiệu tương đối an toàn
    # =========================================================
    out["safe_base_flag"] = (
        (out["is_auto_renew"].fillna(0) == 1)
        & (out["silent_expiring_flag"] == 0)
        & (out["days_since_last_listen"].fillna(np.inf) <= RECENCY_SAFE_DAYS)
        & (out["risk_score"] <= SAFE_BASE_MAX_RISK_SCORE)
    ).astype("int8")

    # =========================================================
    # 10) Revenue decomposition
    # Tách revenue của expiring cohort thành nhiều lớp ý nghĩa:
    # - safe_base_revenue
    # - silent_expiring_revenue
    # - revenue_at_risk_30d
    # - realized_lost_revenue_30d
    # =========================================================
    out["safe_base_revenue"] = out["revenue_up_for_renewal"] * out["safe_base_flag"]
    out["silent_expiring_revenue"] = (
        out["revenue_up_for_renewal"] * out["silent_expiring_flag"]
    )
    out["revenue_at_risk_30d"] = (
        out["revenue_up_for_renewal"] * out["projected_risk_flag"]
    )
    out["realized_lost_revenue_30d"] = (
        out["revenue_up_for_renewal"] * out["churned_flag"]
    )

    # =========================================================
    # 11) User tier
    # Phân tầng theo mức expected renewal amount
    # =========================================================
    out["user_tier"] = np.select(
        [
            out["revenue_up_for_renewal"] <= 0,
            out["revenue_up_for_renewal"] < 100,
            out["revenue_up_for_renewal"] < 150,
            out["revenue_up_for_renewal"] >= 150,
        ],
        ["Free / Trial", "Budget", "Standard", "Premium"],
        default="Unknown",
    )

    # =========================================================
    # 12) Strategic segmentation flags
    # =========================================================

    # High value:
    # - RFM score đủ cao
    # hoặc
    # - expected renewal amount đủ lớn
    out["high_value_flag"] = (
        (out.get("rfm_total_score", pd.Series(0, index=out.index)).fillna(0) >= 8)
        | (out["revenue_up_for_renewal"] >= HIGH_VALUE_AMOUNT)
    ).astype("int8")

    # Loyal:
    # - tenure dài
    # - chu kỳ trước ổn định
    # - kỳ hiện tại vẫn renew
    out["loyal_flag"] = (
        (out["membership_age_days"].fillna(-1) >= LOYAL_TENURE_DAYS)
        & (out["last_1_is_churn"].fillna(-1) == 0)
        & (out["renewed_flag"] == 1)
    ).astype("int8")

    # At risk: tái dùng projected_risk_flag
    out["at_risk_flag"] = out["projected_risk_flag"]

    # =========================================================
    # 13) Customer segment cuối cùng
    # Ưu tiên:
    # 1. New
    # 2. At Risk
    # 3. Loyal
    # 4. High Value
    # 5. còn lại là Core
    # =========================================================
    out["customer_segment"] = np.select(
        [
            out["new_user_flag"] == 1,
            out["at_risk_flag"] == 1,
            out["loyal_flag"] == 1,
            out["high_value_flag"] == 1,
        ],
        ["New", "At Risk", "Loyal", "High Value"],
        default="Core",
    )

    return out


# =========================
# Run feature engineering
# =========================
analysis_df = add_dashboard_features(raw_bi_df)

# =========================
# Mini data dictionary preview
# =========================
engineered_feature_preview = pd.DataFrame(
    {
        "feature": [
            "revenue_up_for_renewal",
            "renewed_flag",
            "churned_flag",
            "observable_active_flag",
            "silent_expiring_flag",
            "customer_composition",
            "previous_customer_state",
            "risk_score",
            "risk_band",
            "projected_risk_flag",
            "safe_base_flag",
            "safe_base_revenue",
            "silent_expiring_revenue",
            "revenue_at_risk_30d",
            "realized_lost_revenue_30d",
            "user_tier",
            "customer_segment",
        ],
        "sample_definition": [
            "clip(expected_renewal_amount, lower=0)",
            "is_churn == 0",
            "is_churn == 1",
            "có usage observable từ active_segment hoặc fallback count/listen > 0",
            "expiring cohort nhưng không còn listening signal rõ ràng",
            "Churned / New / Returning / Retained",
            "First Cycle / Previously Churned / Previously Renewed",
            "Business-rule score 0-6",
            "Low Risk / Watchlist / Medium Risk / High Risk",
            "risk_score >= 3 hoặc silent_expiring_flag == 1",
            "auto-renew + không silent + recency tốt + risk thấp",
            "revenue_up_for_renewal * safe_base_flag",
            "revenue_up_for_renewal * silent_expiring_flag",
            "revenue_up_for_renewal * projected_risk_flag",
            "revenue_up_for_renewal * churned_flag",
            "Free / Trial / Budget / Standard / Premium",
            "New / At Risk / Loyal / High Value / Core",
        ],
    }
)

display(engineered_feature_preview)
print("analysis_df shape:", analysis_df.shape)

,feature,sample_definition
0,revenue_up_for_renewal,"clip(expected_renewal_amount, lower=0)"
1,renewed_flag,is_churn == 0
2,churned_flag,is_churn == 1
3,observable_active_flag,có usage observable từ active_segment hoặc fal...
4,silent_expiring_flag,expiring cohort nhưng không còn listening sign...
5,customer_composition,Churned / New / Returning / Retained
6,previous_customer_state,First Cycle / Previously Churned / Previously ...
7,risk_score,Business-rule score 0-6
8,risk_band,Low Risk / Watchlist / Medium Risk / High Risk
9,projected_risk_flag,risk_score >= 3 hoặc silent_expiring_flag == 1


analysis_df shape: (44162, 148)


# Snapshot analysis theo `target_month` và `target_year`

## Executive KPI Cards

Bộ KPI này tóm tắt **4 chỉ số chính của snapshot tháng đang chọn** để lãnh đạo đọc rất nhanh quy mô cohort và giá trị doanh thu của tháng đó.

- **Expiring Subscribers**: số user duy nhất trong `snapshot_df`.
- **New Paid Users**: tổng `new_user_flag = 1`, tức là user mới bước vào lifecycle trả phí trong tháng.
- **Total Revenue**: tổng `revenue_up_for_renewal`, tức tổng renewal amount đang đến kỳ quyết định trong tháng.
- **APRU**: `Total Revenue / Expiring Subscribers`, dùng để nhìn nhanh giá trị doanh thu bình quân trên mỗi user trong cohort.


In [7]:
snapshot_df = get_snapshot_slice(analysis_df, target_year, target_month)
snapshot_label = snapshot_df['month_label'].iloc[0]

expiring_subscribers = snapshot_df['msno'].nunique()
new_paid_users = int(snapshot_df['new_user_flag'].sum())
total_revenue = float(snapshot_df['revenue_up_for_renewal'].sum())
apru = safe_divide(total_revenue, expiring_subscribers, default=0.0)

print(f'Snapshot month: {snapshot_label}')

display_kpi_cards([
    ('Expiring Subscribers', f'{expiring_subscribers:,.0f}', 'Số lượng user nằm trong expiring cohort của tháng mục tiêu'),
    ('New Paid Users', f'{new_paid_users:,.0f}', 'Số user mới bước vào lifecycle trả phí trong tháng theo `new_user_flag`'),
    ('Total Revenue', format_currency(total_revenue), 'Tổng `revenue_up_for_renewal` của expiring cohort trong tháng'),
    ('APRU', format_currency(apru), 'Average revenue per user = Total Revenue / Expiring Subscribers'),
], n_cols=4)


Snapshot month: 2017-03


## Churn Rate

Chart donut này cho thấy **tỷ lệ churn của expiring cohort trong tháng đang chọn**.

- Mỗi user được gắn `churned_flag = 1` nếu `is_churn = 1`.
- Phần còn lại là `renewed_flag = 1`.
- Con số ở giữa chart là `mean(churned_flag)`, tức **forward churn rate** của tháng đó.

Chart này giúp C-Level trả lời nhanh: trong cohort đến kỳ gia hạn tháng này, **bao nhiêu phần trăm đã không quay lại trong cửa sổ 30 ngày**.


In [8]:

snapshot_df = get_snapshot_slice(analysis_df, target_year, target_month)
snapshot_label = snapshot_df['month_label'].iloc[0]

churn_counts = pd.DataFrame({
    'status': ['Renewed', 'Churned'],
    'users': [int(snapshot_df['renewed_flag'].sum()), int(snapshot_df['churned_flag'].sum())]
})

fig = px.pie(
    churn_counts,
    names='status',
    values='users',
    hole=0.62,
    title=f'Churn Rate in {snapshot_label}'
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=DEFAULT_HEIGHT)
fig.add_annotation(
    text=f"<b>{snapshot_df['churned_flag'].mean():.1%}</b><br>churn rate",
    x=0.5, y=0.5, showarrow=False, font_size=20
)
fig.show()


## Customer Composition (New/Retained/Returning/Churned)

Chart này phân rã expiring cohort theo **`customer_composition`**, để thấy cơ cấu khách hàng của tháng đang chọn gồm những nhóm nào.

Logic gán nhóm:
- **Churned**: user churn trong tháng.
- **New**: user mới bước vào lifecycle trả phí (`new_user_flag = 1`).
- **Returning**: user từng churn trước đó nhưng hiện tại quay lại và renewed.
- **Retained**: user không phải new/returning và vẫn được giữ lại.

Số lượng ở mỗi nhánh là **số user** thuộc từng nhóm trong snapshot.


In [9]:
snapshot_df = get_snapshot_slice(analysis_df, target_year, target_month)
snapshot_label = snapshot_df['month_label'].iloc[0]

composition_df = (
    snapshot_df['customer_composition']
    .fillna('Unknown')
    .value_counts(dropna=False)
    .rename_axis('customer_composition')
    .reset_index(name='users')
    .sort_values('users', ascending=True)
)

fig = px.bar(
    composition_df,
    x='users',
    y='customer_composition',
    orientation='h',
    text='users',
    title=f'Customer Composition in {snapshot_label}'
)

fig.update_traces(
    textposition='outside'
)

fig.update_layout(
    height=DEFAULT_HEIGHT,
    xaxis_title='Number of Users',
    yaxis_title='Customer Composition',
    uniformtext_minsize=10,
    uniformtext_mode='hide'
)

fig.show()

## Customer Segments (High Value/At Risk/New/Loyal)

Treemap này gom user theo **`customer_segment`** để giúp lãnh đạo nhìn nhanh mix chất lượng của cohort.

`customer_segment` được gán theo business rule:
- **New**
- **At Risk**
- **Loyal**
- **High Value**
- **Core**

Diện tích mỗi ô biểu diễn **số user** trong segment. Nhờ đó có thể thấy tháng này cohort đang nghiêng nhiều về nhóm cần bảo vệ, nhóm giá trị cao hay nhóm mới.


In [10]:
snapshot_df = get_snapshot_slice(analysis_df, target_year, target_month)
snapshot_label = snapshot_df['month_label'].iloc[0]

segment_df = (
    snapshot_df['customer_segment']
    .fillna('Unknown')
    .value_counts(dropna=False)
    .rename_axis('customer_segment')
    .reset_index(name='users')
    .sort_values('users', ascending=False)
)

fig = px.treemap(
    segment_df,
    path=['customer_segment'],
    values='users',
    color='users',
    color_continuous_scale=[
        [0.00, '#dbeafe'],   # ít -> xanh rất nhạt
        [0.35, '#93c5fd'],
        [0.65, '#3b82f6'],
        [1.00, '#1d4ed8']    # nhiều -> xanh đậm
    ],
    title=f'Customer Segments in {snapshot_label}'
)

fig.update_traces(
    texttemplate="<b>%{value:,}</b><br>%{label}",
    textfont_size=28,
    textfont_color="#0F172A",
    textposition="middle center",    
    hovertemplate="<b>%{label}</b><br>Users: %{value:,}<extra></extra>",
    marker=dict(
        line=dict(color="white", width=3)
    ),
    root_color="white"
)

fig.update_layout(
    height=DEFAULT_HEIGHT,
    margin=dict(t=60, l=10, r=10, b=10),
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(color="black"),
    uniformtext_minsize=11,
    uniformtext_mode="hide",
    coloraxis_showscale=False
)

fig.show()

## Subscriber Movement

Sankey chart này cho thấy **dòng dịch chuyển từ trạng thái trước đó sang trạng thái hiện tại** của user trong expiring cohort.

- Nút bên trái là `previous_customer_state`.
- Nút bên phải là `customer_composition`.
- Độ dày của luồng là **số user duy nhất (`nunique(msno)`)** nằm trong cặp chuyển dịch đó.

Chart này hữu ích để trả lời câu hỏi kiểu: *khách previously renewed kỳ này chủ yếu retained hay churned?* hoặc *khách first cycle đang rơi vào nhóm nào?*


In [11]:

snapshot_df = get_snapshot_slice(analysis_df, target_year, target_month)
snapshot_label = snapshot_df['month_label'].iloc[0]

movement_df = (
    snapshot_df.groupby(['previous_customer_state', 'customer_composition'], as_index=False)
    .agg(users=('msno', 'nunique'))
)

source_labels = movement_df['previous_customer_state'].tolist()
target_labels = movement_df['customer_composition'].tolist()
all_labels = list(dict.fromkeys(source_labels + target_labels))
label_to_idx = {label: idx for idx, label in enumerate(all_labels)}

fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=18,
        thickness=18,
        label=all_labels,
    ),
    link=dict(
        source=[label_to_idx[x] for x in movement_df['previous_customer_state']],
        target=[label_to_idx[x] for x in movement_df['customer_composition']],
        value=movement_df['users']
    )
)])
fig.update_layout(title_text=f'Subscriber Movement in {snapshot_label}', height=WIDE_HEIGHT)
fig.show()


## So sánh Revenue hiện tại: Expiring vs Safe Base

Chart dưới đây đọc revenue theo đúng logic của `features-prep.ipynb` và lớp business rule trong notebook:

- **Revenue Up for Renewal** = tổng `expected_renewal_amount` của expiring cohort trong tháng.
- **Safe Renewal Base** = `revenue_up_for_renewal * safe_base_flag`.
- **Silent / Inactive Revenue** = `revenue_up_for_renewal * silent_expiring_flag`.
- **Projected Revenue at Risk 30D** = `revenue_up_for_renewal * projected_risk_flag`.
- **Realized Lost Revenue 30D** = `revenue_up_for_renewal * churned_flag`.

Notebook dùng **bar chart** thay vì funnel vì đây là các lớp revenue business để so sánh quy mô, không phải chuỗi bước tuần tự loại trừ lẫn nhau.


In [12]:
snapshot_df = get_snapshot_slice(analysis_df, target_year, target_month)
snapshot_label = snapshot_df['month_label'].iloc[0]

summary_df = pd.DataFrame({
    'stage': [
        'Revenue Up for Renewal',
        'Safe Renewal Base',
        'Silent / Inactive Revenue',
        'Projected Revenue at Risk 30D',
        'Realized Lost Revenue 30D',
    ],
    'value': [
        float(snapshot_df['revenue_up_for_renewal'].sum()),
        float(snapshot_df['safe_base_revenue'].sum()),
        float(snapshot_df['silent_expiring_revenue'].sum()),
        float(snapshot_df['revenue_at_risk_30d'].sum()),
        float(snapshot_df['realized_lost_revenue_30d'].sum()),
    ]
})
summary_df['share_of_up_for_renewal'] = safe_divide(
    summary_df['value'], summary_df.loc[summary_df['stage'] == 'Revenue Up for Renewal', 'value'].iloc[0]
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=summary_df['stage'],
    y=summary_df['value'],
    text=summary_df['value'].map(lambda x: format_currency(x)),
    textposition='outside',
    customdata=np.stack([summary_df['share_of_up_for_renewal']], axis=1),
    hovertemplate=(
        'Metric: %{x}<br>'
        'Value: %{y:,.0f}<br>'
        'Share of revenue up for renewal: %{customdata[0]:.1%}<extra></extra>'
    )
))
fig.update_layout(
    title=f'Revenue Structure Within Expiring Cohort in {snapshot_label}',
    xaxis_title='Revenue layer',
    yaxis_title='Amount',
    height=DEFAULT_HEIGHT,
)
fig.show()

summary_df


,stage,value,share_of_up_for_renewal
0,Revenue Up for Renewal,"1,848,634.50",1.00
1,Safe Renewal Base,"1,002,860.75",0.00
2,Silent / Inactive Revenue,"18,013.40",0.00
3,Projected Revenue at Risk 30D,"132,635.81",0.00
4,Realized Lost Revenue 30D,"203,720.75",0.00


## Pareto Revenue Distribution by Decile

Chart này giúp kiểm tra mức độ **tập trung doanh thu** trong expiring cohort của tháng đang chọn.

Cách tính:
- Sắp xếp user theo `revenue_up_for_renewal`.
- Chia user thành 10 nhóm theo thứ hạng doanh thu.
- Tính doanh thu từng decile và phần trăm doanh thu tích luỹ.

Đường cumulative share cho biết liệu một số ít user có đang nắm phần lớn revenue hay không.


In [13]:
snapshot_df = get_snapshot_slice(analysis_df, target_year, target_month).copy()
snapshot_label = snapshot_df['month_label'].iloc[0]

pareto = (
    snapshot_df.loc[snapshot_df['revenue_up_for_renewal'] > 0, ['msno', 'revenue_up_for_renewal']]
    .sort_values('revenue_up_for_renewal', ascending=False)
    .reset_index(drop=True)
)

if pareto.empty:
    raise ValueError('Không có revenue_up_for_renewal > 0 để tính Pareto.')

pareto['rank'] = np.arange(1, len(pareto) + 1)
pareto['revenue_decile'] = np.ceil(pareto['rank'] / len(pareto) * 10).astype(int).clip(1, 10)

pareto_summary = (
    pareto.groupby('revenue_decile', as_index=False)
    .agg(
        decile_revenue=('revenue_up_for_renewal', 'sum'),
        users=('msno', 'nunique')
    )
    .sort_values('revenue_decile')
)

total_revenue = pareto_summary['decile_revenue'].sum()
pareto_summary['revenue_share'] = pareto_summary['decile_revenue'] / total_revenue
pareto_summary['cumulative_revenue_share'] = pareto_summary['revenue_share'].cumsum()
pareto_summary['decile_label'] = 'D' + pareto_summary['revenue_decile'].astype(str)

fig = make_subplots(specs=[[{'secondary_y': True}]])

# Bar: revenue by decile
fig.add_trace(
    go.Bar(
        x=pareto_summary['decile_label'],
        y=pareto_summary['decile_revenue'],
        name='Revenue by decile',
        text=pareto_summary['decile_revenue'].map(lambda x: f'{x:,.0f}'),
        textposition='outside',
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Revenue: %{y:,.0f}<br>'
            'Users: %{customdata:,}'
            '<extra></extra>'
        ),
        customdata=pareto_summary['users'],
        marker_line_width=0
    ),
    secondary_y=False
)

# Line: cumulative revenue share
fig.add_trace(
    go.Scatter(
        x=pareto_summary['decile_label'],
        y=pareto_summary['cumulative_revenue_share'],
        mode='lines+markers+text',
        name='Cumulative revenue share',
        text=pareto_summary['cumulative_revenue_share'].map(lambda x: f'{x:.0%}'),
        textposition='top center',
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Cumulative share: %{y:.1%}'
            '<extra></extra>'
        )
    ),
    secondary_y=True
)

# 80% reference line
fig.add_hline(
    y=0.8,
    line_dash='dash',
    line_width=2,
    secondary_y=True,
    annotation_text='80%',
    annotation_position='top right'
)

fig.update_yaxes(
    title_text='Revenue up for renewal',
    secondary_y=False,
    rangemode='tozero',
    showgrid=False
)

fig.update_yaxes(
    title_text='Cumulative revenue share',
    secondary_y=True,
    tickformat='.0%',
    range=[0, 1.05]
)

fig.update_xaxes(title_text='Revenue decile')

fig.update_layout(
    title=f'Pareto Revenue Distribution in {snapshot_label}',
    height=DEFAULT_HEIGHT,
    bargap=0.25,
    hovermode='x unified',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='left',
        x=0
    ),
    margin=dict(t=80, l=60, r=60, b=40)
)

fig.show()

pareto_summary

,revenue_decile,decile_revenue,users,revenue_share,cumulative_revenue_share,decile_label
0,1,"380,359.22",1167,0.21,0.21,D1
1,2,"232,288.56",1167,0.13,0.33,D2
2,3,"203,260.06",1167,0.11,0.44,D3
3,4,"173,883.00",1167,0.09,0.54,D4
4,5,"174,032.00",1168,0.09,0.63,D5
5,6,"173,804.88",1167,0.09,0.72,D6
6,7,"159,680.48",1167,0.09,0.81,D7
7,8,"145,338.89",1167,0.08,0.89,D8
8,9,"115,418.27",1167,0.06,0.95,D9
9,10,"90,569.20",1168,0.05,1.00,D10


## Projected Revenue at Risk 30 ngày tới theo user_tier

Chart này phân rã **revenue rủi ro dự báo** theo từng `user_tier`, để biết tier nào đang cần ưu tiên giữ chân.

Cách tính trong từng tier:
- `revenue_up_for_renewal`: tổng doanh thu sắp đến kỳ gia hạn.
- `projected_revenue_at_risk_30d`: tổng `revenue_at_risk_30d`.
- `risk_share = projected_revenue_at_risk_30d / revenue_up_for_renewal`.
- `silent_expiring_revenue`: phần doanh thu không còn usage observable rõ ràng.

Điểm dữ liệu trên chart là **projected revenue at risk**, còn hover giúp đọc thêm quy mô user và tỷ trọng rủi ro của từng tier.


In [14]:
snapshot_df = get_snapshot_slice(analysis_df, target_year, target_month)
snapshot_label = snapshot_df['month_label'].iloc[0]

tier_order = ['Free / Trial', 'Budget', 'Standard', 'Premium']

tier_df = (
    snapshot_df.groupby('user_tier', as_index=False)
    .agg(
        revenue_up_for_renewal=('revenue_up_for_renewal', 'sum'),
        projected_revenue_at_risk_30d=('revenue_at_risk_30d', 'sum'),
        silent_expiring_revenue=('silent_expiring_revenue', 'sum'),
        users=('msno', 'nunique')
    )
)

tier_df['risk_share'] = np.where(
    tier_df['revenue_up_for_renewal'] > 0,
    tier_df['projected_revenue_at_risk_30d'] / tier_df['revenue_up_for_renewal'],
    0
)

tier_df['safer_revenue'] = (
    tier_df['revenue_up_for_renewal'] - tier_df['projected_revenue_at_risk_30d']
).clip(lower=0)

tier_df['order'] = tier_df['user_tier'].map({k: i for i, k in enumerate(tier_order)})
tier_df = (
    tier_df.sort_values('order')
    .reset_index(drop=True)
)

fig = go.Figure()

# Phần revenue còn lại chưa at risk
fig.add_trace(
    go.Bar(
        y=tier_df['user_tier'],
        x=tier_df['safer_revenue'],
        orientation='h',
        name='Lower-risk renewal revenue',
        marker=dict(color='#BFDBFE'),
        customdata=np.stack([
            tier_df['revenue_up_for_renewal'],
            tier_df['projected_revenue_at_risk_30d'],
            tier_df['users'],
            tier_df['risk_share'],
            tier_df['silent_expiring_revenue']
        ], axis=1),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Revenue up for renewal: %{customdata[0]:,.0f}<br>'
            'Projected revenue at risk 30D: %{customdata[1]:,.0f}<br>'
            'Users: %{customdata[2]:,.0f}<br>'
            'Risk share: %{customdata[3]:.1%}<br>'
            'Silent / inactive revenue: %{customdata[4]:,.0f}'
            '<extra></extra>'
        )
    )
)

# Phần revenue at risk
fig.add_trace(
    go.Bar(
        y=tier_df['user_tier'],
        x=tier_df['projected_revenue_at_risk_30d'],
        orientation='h',
        name='Projected revenue at risk 30D',
        marker=dict(color='#2563EB'),
        text=tier_df['projected_revenue_at_risk_30d'].map(lambda x: f'{x:,.0f}'),
        textposition='inside',
        insidetextanchor='middle',
        textfont=dict(color='white', size=13),
        customdata=np.stack([
            tier_df['revenue_up_for_renewal'],
            tier_df['users'],
            tier_df['risk_share'],
            tier_df['silent_expiring_revenue']
        ], axis=1),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Projected revenue at risk 30D: %{x:,.0f}<br>'
            'Revenue up for renewal: %{customdata[0]:,.0f}<br>'
            'Users: %{customdata[1]:,.0f}<br>'
            'Risk share: %{customdata[2]:.1%}<br>'
            'Silent / inactive revenue: %{customdata[3]:,.0f}'
            '<extra></extra>'
        )
    )
)

# Gắn % risk share ở cuối mỗi bar
max_revenue = tier_df['revenue_up_for_renewal'].max()
label_offset = max_revenue * 0.04  # tăng/giảm số này nếu muốn xa hơn hoặc gần hơn

fig.add_trace(
    go.Scatter(
        x=tier_df['revenue_up_for_renewal'] + label_offset,
        y=tier_df['user_tier'],
        mode='text',
        text=tier_df['risk_share'].map(lambda x: f'{x:.1%}'),
        textfont=dict(size=13, color='#1E3A8A'),
        showlegend=False,
        hoverinfo='skip'
    )
)

fig.update_layout(
    title=f'Projected Revenue at Risk 30D by User Tier in {snapshot_label}',
    barmode='stack',
    height=DEFAULT_HEIGHT,
    xaxis_title='Revenue',
    yaxis_title='User Tier',
    bargap=0.35,
    hovermode='y unified',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='left',
        x=0
    ),
    margin=dict(t=80, l=80, r=80, b=40),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig.update_xaxes(showgrid=True, gridcolor='rgba(0,0,0,0.08)', zeroline=False)
fig.update_yaxes(categoryorder='array', categoryarray=tier_order)

fig.show()

tier_df[[
    'user_tier', 'users', 'revenue_up_for_renewal',
    'projected_revenue_at_risk_30d', 'safer_revenue',
    'silent_expiring_revenue', 'risk_share'
]]

,user_tier,users,revenue_up_for_renewal,projected_revenue_at_risk_30d,safer_revenue,silent_expiring_revenue,risk_share
0,Free / Trial,427,0.00,0.00,0.00,0.00,0.00
1,Budget,2363,"208,759.47","27,531.03","181,228.44","3,324.02",0.13
2,Standard,6072,"863,306.19","56,709.76","806,596.44","10,291.95",0.07
3,Premium,3237,"776,568.94","48,395.02","728,173.94","4,397.43",0.06


# Trend analysis theo giai đoạn cấu hình

## Executive KPI Cards

Bộ KPI này giữ nguyên 4 chỉ số chính nhưng đọc ở **mức giai đoạn trend** thay vì một tháng đơn lẻ.

- **Expiring Subscribers**: tổng quy mô expiring cohort cộng dồn qua các tháng trong giai đoạn.
- **New Paid Users**: tổng user mới (`new_user_flag = 1`) cộng dồn qua các tháng.
- **Total Revenue**: tổng `revenue_up_for_renewal` của toàn giai đoạn.
- **APRU**: `Total Revenue / Expiring Subscribers` trên toàn giai đoạn đã chọn.

Nhờ vậy phần trend vẫn bám đúng bộ KPI chính, nhưng diễn giải theo logic tổng hợp nhiều tháng.


In [15]:
trend_df = get_trend_slice(analysis_df, year_start, month_start, year_end, month_end)
monthly_summary = build_monthly_summary(trend_df)
period_label_text = f"{month_label(monthly_summary['target_month'].min())} → {month_label(monthly_summary['target_month'].max())}"

period_expiring_subscribers = float(monthly_summary['expiring_subscribers'].sum())
period_new_paid_users = float(monthly_summary['new_users'].sum())
period_total_revenue = float(monthly_summary['revenue_up_for_renewal'].sum())
period_apru = safe_divide(
    period_total_revenue,
    period_expiring_subscribers,
    default=0.0
)

print(f'Trend range: {period_label_text}')

display_kpi_cards([
    ('Expiring Subscribers', f'{period_expiring_subscribers:,.0f}', 'Tổng quy mô expiring cohort cộng dồn qua các tháng trong giai đoạn đã chọn'),
    ('New Paid Users', f'{period_new_paid_users:,.0f}', 'Tổng số user mới bước vào lifecycle trả phí trong toàn giai đoạn'),
    ('Total Revenue', format_currency(period_total_revenue), 'Tổng `revenue_up_for_renewal` của expiring cohort trong giai đoạn'),
    ('APRU', format_currency(period_apru), 'Average revenue per user của giai đoạn = Total Revenue / Expiring Subscribers'),
], n_cols=4)


Trend range: 2017-01 → 2017-04


## Customer Base Growth Trend (Expiring Cohort)

Vì `features-prep.ipynb` build theo **expiring cohort**, chart này theo dõi tăng trưởng của **quy mô cohort đáo hạn theo tháng**, không phải full active base của công ty.

- Trục trái: `expiring_subscribers` theo từng tháng.
- Trục phải: `expiring_cohort_growth_rate = pct_change(expiring_subscribers)`.

Biểu đồ này cho thấy cohort đáo hạn đang mở rộng hay co lại qua thời gian.


In [16]:
trend_df = get_trend_slice(analysis_df, year_start, month_start, year_end, month_end)
monthly_summary = build_monthly_summary(trend_df).copy()

monthly_summary['month_label'] = pd.to_datetime(monthly_summary['analysis_month']).dt.strftime('%b %Y')
monthly_summary['growth_label'] = monthly_summary['expiring_cohort_growth_rate'].map(
    lambda x: f'{x:.1%}' if pd.notnull(x) else ''
)

fig = make_subplots(specs=[[{'secondary_y': True}]])

# Bar: expiring subscribers
fig.add_trace(
    go.Bar(
        x=monthly_summary['month_label'],
        y=monthly_summary['expiring_subscribers'],
        name='Expiring subscribers',
        text=monthly_summary['expiring_subscribers'].map(lambda x: f'{x:,.0f}'),
        textposition='outside',
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Expiring subscribers: %{y:,.0f}'
            '<extra></extra>'
        )
    ),
    secondary_y=False
)

# Line: MoM growth rate
fig.add_trace(
    go.Scatter(
        x=monthly_summary['month_label'],
        y=monthly_summary['expiring_cohort_growth_rate'],
        mode='lines+markers+text',
        name='MoM growth rate',
        text=monthly_summary['growth_label'],
        textposition='top center',
        hovertemplate=(
            '<b>%{x}</b><br>'
            'MoM growth rate: %{y:.1%}'
            '<extra></extra>'
        )
    ),
    secondary_y=True
)

fig.update_yaxes(
    title_text='Expiring subscribers',
    secondary_y=False,
    rangemode='tozero'
)

fig.update_yaxes(
    title_text='Growth rate',
    secondary_y=True,
    tickformat='.0%',
    zeroline=False
)

fig.update_xaxes(title_text='Month')

fig.update_layout(
    title='Customer Base Growth Trend (Expiring Cohort)',
    height=DEFAULT_HEIGHT,
    hovermode='x unified',
    bargap=0.35,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='left',
        x=0
    ),
    margin=dict(t=80, l=60, r=60, b=40)
)

fig.show()

## Revenue generation vs. Short-term Risk Trend

Trend này đặt cạnh nhau bốn lớp revenue quan trọng của từng tháng:

- **Revenue Up for Renewal**
- **Safe Renewal Base**
- **Projected Revenue at Risk 30D**
- **Realized Lost Revenue 30D**

Tất cả đều được tổng hợp từ `monthly_summary`, tức là **sum theo tháng** trên expiring cohort. Nhờ đó C-Level có thể đọc cùng lúc quy mô doanh thu, phần an toàn, phần cần bảo vệ, và phần đã mất thực tế.


In [17]:
trend_df = get_trend_slice(analysis_df, year_start, month_start, year_end, month_end)
monthly_summary = build_monthly_summary(trend_df).copy()

monthly_summary['month_label'] = pd.to_datetime(
    monthly_summary['analysis_month']
).dt.strftime('%b %Y')

def fmt_short(x):
    if pd.isna(x):
        return ''
    x = float(x)
    if abs(x) >= 1_000_000:
        return f'{x/1_000_000:.2f}M'
    if abs(x) >= 1_000:
        return f'{x/1_000:.0f}k'
    return f'{x:,.0f}'

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=monthly_summary['month_label'],
        y=monthly_summary['safe_base_revenue'],
        name='Safe Renewal Base',
        text=monthly_summary['safe_base_revenue'].map(fmt_short),
        textposition='inside',
        insidetextanchor='middle',
        textfont=dict(color='white', size=13),
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Safe Renewal Base: %{y:,.0f}'
            '<extra></extra>'
        )
    )
)

fig.add_trace(
    go.Bar(
        x=monthly_summary['month_label'],
        y=monthly_summary['revenue_at_risk_30d'],
        name='Projected Revenue at Risk 30D',
        text=monthly_summary['revenue_at_risk_30d'].map(fmt_short),
        textposition='inside',
        insidetextanchor='middle',
        textfont=dict(color='white', size=13),
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Projected Revenue at Risk 30D: %{y:,.0f}'
            '<extra></extra>'
        )
    )
)

fig.add_trace(
    go.Bar(
        x=monthly_summary['month_label'],
        y=monthly_summary['realized_lost_revenue_30d'],
        name='Realized Lost Revenue 30D',
        text=monthly_summary['realized_lost_revenue_30d'].map(fmt_short),
        textposition='inside',
        insidetextanchor='middle',
        textfont=dict(color='white', size=13),
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Realized Lost Revenue 30D: %{y:,.0f}'
            '<extra></extra>'
        )
    )
)

fig.update_layout(
    title='Revenue Generation vs Short-term Risk Trend',
    height=DEFAULT_HEIGHT,
    barmode='stack',
    xaxis_title='Month',
    yaxis_title='Revenue',
    hovermode='x unified',
    bargap=0.35,
    uniformtext_minsize=10,
    uniformtext_mode='hide',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='left',
        x=0
    ),
    margin=dict(t=80, l=60, r=40, b=40),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig.update_yaxes(
    rangemode='tozero',
    showgrid=True,
    gridcolor='rgba(0,0,0,0.08)'
)

fig.show()

## New vs Churned Users by Month

Chart này so sánh hai dòng user count theo tháng:

- **New users** = tổng `new_user_flag`.
- **Churned users** = tổng `churned_flag`.

Đây là cách nhìn rất nhanh để xem tháng nào lượng khách mới vào hệ thống có đủ bù cho lượng khách rời bỏ hay không.


In [18]:
trend_df = get_trend_slice(analysis_df, year_start, month_start, year_end, month_end)
monthly_summary = build_monthly_summary(trend_df).copy()

monthly_summary['month_label'] = pd.to_datetime(
    monthly_summary['analysis_month']
).dt.strftime('%b %Y')

monthly_summary['churned_users_neg'] = -monthly_summary['churned_users']
monthly_summary['net_user_change'] = monthly_summary['new_users'] - monthly_summary['churned_users']

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=monthly_summary['month_label'],
        y=monthly_summary['new_users'],
        name='New users',
        text=monthly_summary['new_users'].map(lambda x: f'{x:,.0f}'),
        textposition='outside',
        hovertemplate=(
            '<b>%{x}</b><br>'
            'New users: %{y:,.0f}<extra></extra>'
        )
    )
)

fig.add_trace(
    go.Bar(
        x=monthly_summary['month_label'],
        y=monthly_summary['churned_users_neg'],
        name='Churned users',
        text=monthly_summary['churned_users'].map(lambda x: f'{x:,.0f}'),
        textposition='outside',
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Churned users: %{customdata:,.0f}<extra></extra>'
        ),
        customdata=monthly_summary['churned_users']
    )
)

fig.add_hline(y=0, line_width=1.5, line_color='gray')

fig.update_layout(
    title='New vs Churned Users by Month',
    height=DEFAULT_HEIGHT,
    barmode='relative',
    xaxis_title='Month',
    yaxis_title='Users (- churned / + new)',
    hovermode='x unified',
    bargap=0.35,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='left',
        x=0
    ),
    margin=dict(t=80, l=60, r=40, b=40),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig.update_yaxes(
    tickformat=',',
    zeroline=False,
    showgrid=True,
    gridcolor='rgba(0,0,0,0.08)'
)

fig.show()

## Forward Churn Rate Trend (From Expiring Cohort)

Biểu đồ này theo dõi **forward churn rate** của expiring cohort qua thời gian.

- `forward_churn_rate = churned_users / expiring_subscribers`
- Đường thứ hai là **3-month moving average** để làm mượt nhiễu ngắn hạn.

Chart này phù hợp cho lãnh đạo vì nó cho thấy xu hướng churn nền đang đi lên, đi ngang hay cải thiện dần.


In [19]:
trend_df = get_trend_slice(analysis_df, year_start, month_start, year_end, month_end)
monthly_summary = build_monthly_summary(trend_df).copy()
monthly_summary['forward_churn_rate_3m_ma'] = (
    monthly_summary['forward_churn_rate']
    .rolling(3, min_periods=1)
    .mean()
)

monthly_summary['month_label'] = pd.to_datetime(
    monthly_summary['analysis_month']
).dt.strftime('%b %Y')

rate_cols = ['forward_churn_rate', 'forward_churn_rate_3m_ma']
y_min = monthly_summary[rate_cols].min().min()
y_max = monthly_summary[rate_cols].max().max()
padding = max((y_max - y_min) * 0.30, 0.002)

actual_text = [''] * len(monthly_summary)
ma_text = [''] * len(monthly_summary)
actual_text[-1] = f"{monthly_summary['forward_churn_rate'].iloc[-1]:.2%}"
ma_text[-1] = f"{monthly_summary['forward_churn_rate_3m_ma'].iloc[-1]:.2%}"

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=monthly_summary['month_label'],
        y=monthly_summary['forward_churn_rate'],
        mode='lines+markers+text',
        name='Forward churn rate',
        text=actual_text,
        textposition='top right',
        line=dict(color='#5B8DEF', width=3),
        marker=dict(size=9),
        hovertemplate='<b>%{x}</b><br>Forward churn rate: %{y:.2%}<extra></extra>'
    )
)

fig.add_trace(
    go.Scatter(
        x=monthly_summary['month_label'],
        y=monthly_summary['forward_churn_rate_3m_ma'],
        mode='lines+markers+text',
        name='3M moving average',
        text=ma_text,
        textposition='bottom right',
        line=dict(color='#F05A3E', width=3, dash='dash'),
        marker=dict(size=7),
        hovertemplate='<b>%{x}</b><br>3M moving average: %{y:.2%}<extra></extra>'
    )
)

fig.update_layout(
    title='Forward Churn Rate Trend (From Expiring Cohort)',
    height=DEFAULT_HEIGHT,
    xaxis_title='Month',
    yaxis_title='Churn rate',
    hovermode='x unified',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='left',
        x=0
    ),
    margin=dict(t=80, l=60, r=40, b=40),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig.update_yaxes(
    tickformat='.1%',
    range=[max(0, y_min - padding), y_max + padding],
    showgrid=True,
    gridcolor='rgba(0,0,0,0.08)',
    zeroline=False
)

fig.show()